In [32]:
import tensorflow as tf 
import keras 

In [19]:
c= tf.Variable([[1,2,3],[4,5,6]] , dtype=float)



In [31]:
indices = [[0, 1], 
           [2, 3]]
values = [5, 10]
dense_shape = [3, 4]

sparse_tensor = tf.SparseTensor(
    indices=indices,
    values=values,
    dense_shape=dense_shape
)

print(sparse_tensor)

SparseTensor(indices=tf.Tensor(
[[0 1]
 [2 3]], shape=(2, 2), dtype=int64), values=tf.Tensor([ 5 10], shape=(2,), dtype=int32), dense_shape=tf.Tensor([3 4], shape=(2,), dtype=int64))


In [33]:
def huber_fn(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1
    squared_loss = tf.square(error) / 2
    linear_loss  = tf.abs(error) - 0.5
    return tf.where(is_small_error, squared_loss, linear_loss)

In [34]:
class HuberLoss(keras.losses.Loss):
    def __init__(self, threshold=1.0, **kwargs):
        self.threshold = threshold
        super().__init__(**kwargs)
    def call(self, y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < self.threshold
        squared_loss = tf.square(error) / 2
        linear_loss  = self.threshold * tf.abs(error) - self.threshold**2 / 2
        return tf.where(is_small_error, squared_loss, linear_loss)
    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "threshold": self.threshold}

In [35]:
class MyDense(keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)
    def build(self, batch_input_shape):
        self.kernel = self.add_weight(
        name="kernel", shape=[batch_input_shape[-1], self.units],
        initializer="glorot_normal")
        self.bias = self.add_weight(
        name="bias", shape=[self.units], initializer="zeros")
        super().build(batch_input_shape) # must be at the end
    def call(self, X):
        return self.activation(X @ self.kernel + self.bias)
    def compute_output_shape(self, batch_input_shape):
        return tf.TensorShape(batch_input_shape.as_list()[:-1] + [self.units])
    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "units": self.units,
        "activation": keras.activations.serialize(self.activation)}

In [37]:
X_sample = tf.random.normal(shape=(4, 5))
# Create a layer with 3 output neurons
my_layer = MyDense(units=3, activation="relu")
# When the layer receives data for the first time, it calls build() automatically:
output = my_layer(X_sample)

print("\nLayer Weights:")
print("Kernel (Weights) shape:", my_layer.kernel.shape)
print("Bias shape:            ", my_layer.bias.shape)
print("Output shape:          ", output.shape)


Layer Weights:
Kernel (Weights) shape: (5, 3)
Bias shape:             (3,)
Output shape:           (4, 3)


In [39]:
class ResidualBlock(keras.layers.Layer):
    def __init__(self , n_layers , n_neurons , **kwargs):
        super().__init__(**kwargs)
        self.hidden = [keras.layers.Dense(n_neurons , activation='elu', kernel_initializer='he_normal'
                                          ) for _ in range (n_layers)]
    def __call__(self , inputs):
        x= inputs 
        for layer in self.hidden:
            x= layer(x)
        return inputs + x 

In [40]:
class ResidualRegressor(keras.Model):
    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.hidden1 = keras.layers.Dense(30, activation="elu",
        kernel_initializer="he_normal")
        self.block1 = ResidualBlock(2, 30)
        self.block2 = ResidualBlock(2, 30)
        self.out = keras.layers.Dense(output_dim)
    def call(self, inputs):
        Z = self.hidden1(inputs)
        for _ in range(1 + 3):
            Z = self.block1(Z)
        Z = self.block2(Z)
        return self.out(Z)

In [ ]:
class ReconstructingRegressor(keras.Model):
    def __init__(self, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.hidden = [keras.layers.Dense(30, activation="selu",
        kernel_initializer="lecun_normal")
                     for _ in range(5)]
        self.out = keras.layers.Dense(output_dim)
    def build(self, batch_input_shape):
        n_inputs = batch_input_shape[-1]
        self.reconstruct = keras.layers.Dense(n_inputs)
        super().build(batch_input_shape)
    def call(self, inputs):
        Z = inputs
        for layer in self.hidden:
            Z = layer(Z)
        reconstruction = self.reconstruct(Z)
        recon_loss = tf.reduce_mean(tf.square(reconstruction - inputs))
        self.add_loss(0.05 * recon_loss)
        return self.out(Z)